# MP3 AI Türkçe Dublaj — Colab
**Runtime > Change runtime type > GPU** seçin.

## 0. (Opsiyonel) Colab Secrets
Repo public; clone için token GEREKMEZ. İstersen 🔑 (anahtar) simgesinden secret ekleyebilirsin (Notebook access açık olsun):
- `HF_TOKEN` → (opsiyonel) pyannote için. Eklemezsen speechbrain kullanılır.
- `GITHUB_TOKEN` → sadece 8. hücrede (sonucu repoya geri push) gerekir.

## 1. Repo + sistem bağımlılıkları

In [ ]:
!apt-get -qq install -y ffmpeg
!pip install -q yt-dlp
%cd /content
!rm -rf dublaj
!git clone https://github.com/0xR0/dublaj.git
%cd dublaj

## 2. Python bağımlılıkları (birkaç dakika)

In [ ]:
!pip install -q -r requirements.txt

## 3. Ortam değişkenleri
HF_TOKEN secret'ı varsa pyannote, yoksa speechbrain kullanılır.

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN') or ''
except Exception:
    os.environ['HF_TOKEN'] = ''
os.environ['COQUI_TOS_AGREED'] = '1'
print('HF_TOKEN ayarli mi:', bool(os.environ['HF_TOKEN']))

## 4-A. Girdi: YouTube linkinden ses indir

In [ ]:
YT_URL = "https://youtu.be/2hsSEWguOQE"  # <-- linki degistir
!yt-dlp -x --audio-format mp3 --audio-quality 0 -o "input/yt_audio.%(ext)s" "$YT_URL"
INPUT = "input/yt_audio.mp3"
print("Girdi:", INPUT)

## 4-B. (Alternatif) Kendi dosyanı yükle

In [ ]:
from google.colab import files
import shutil
up = files.upload()
name = list(up.keys())[0]
shutil.move(name, f'input/{name}')
INPUT = f'input/{name}'
print('Girdi:', INPUT)

## 5. Dublajı çalıştır
İlk sefer modeller iner (XTTS ~1.8GB). Bayraklar: `--no-background`, `--tts edge`, `--diarizer speechbrain|pyannote`.

In [ ]:
!python dub.py "$INPUT"

## 6. Sonucu incele

In [ ]:
!ls -la output/
print('--- run.log (son 40 satir) ---')
!tail -n 40 logs/run.log

## 7. Sonucu indir

In [ ]:
from google.colab import files
files.download('output/output_dubbed.mp3')

## 8. (Opsiyonel) Sonucu GitHub'a geri gönder
Termux tarafında commitler + log + çıktı incelenebilsin diye. Bunun için `GITHUB_TOKEN` secret'i (repo write yetkili) gerekir.

In [ ]:
from google.colab import userdata
GITHUB_TOKEN = userdata.get("GITHUB_TOKEN")
!git config user.email "colab@dublaj"
!git config user.name "colab"
!git add -f output/output_dubbed.mp3 logs/run.log
!git commit -m "colab: dublaj sonucu + loglar" || echo "degisiklik yok"
!git push https://{GITHUB_TOKEN}@github.com/0xR0/dublaj.git HEAD:colab-results